# **LAB Deep Learning Session 5**
- ini cell 1-10 codingan sesi 5 kemarin
- sesi 5 = encode
- sesi 6 = decode

In [3]:
import tensorflow as tf

In [4]:
dataset = tf.data.TextLineDataset("dataset/ind.txt")

In [5]:
VOCAB_SIZE = 7000      
ENG_SEQ_LENGTH = 16
IND_SEQ_LENGTH = 16    
EMBEDDING_DIM = 256
BATCH_SIZE = 128       
HIDDEN_UNITS = 256     

In [6]:
eng_vec_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=ENG_SEQ_LENGTH
)

ind_vec_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=IND_SEQ_LENGTH
)

In [7]:
def preprocess(text):
    text = tf.strings.split(text, "\t")
    eng = text[0]
    ind = "starttoken " + text[1] + " endtoken"
    return eng, ind

def vectorize(text):
    text = tf.strings.split(text, "\t")
    eng = text[0]
    start = "starttoken " + text[1]     
    end = text[1] + " endtoken"         
    return {
        "input_1": eng_vec_layer(eng),      
        "input_2": ind_vec_layer(start),    
    }, ind_vec_layer(end)

In [8]:
preprocessed_dataset = dataset.map(preprocess)

eng_adapt_dataset = preprocessed_dataset.map(lambda eng, ind: eng).batch(1024)
eng_vec_layer.adapt(eng_adapt_dataset)

ind_adapt_dataset = preprocessed_dataset.map(lambda eng, ind: ind).batch(1024)
ind_vec_layer.adapt(ind_adapt_dataset)

In [9]:
n_data = int(dataset.reduce(0, lambda count, _: count + 1))   
n_val = int(n_data * 0.2)                                     

shuffled = dataset.shuffle(VOCAB_SIZE, seed=8)
val_raw, train_raw = shuffled.take(n_val), shuffled.skip(n_val)

train_dataset = (train_raw.map(vectorize).cache().shuffle(4096).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_raw.map(vectorize).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [10]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, hidden_units):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)  
        self.LSTM = tf.keras.layers.LSTM(hidden_units, return_sequences=True)

    def call(self, x):
        x = self.embedding(x)     
        return self.LSTM(x)

# **Bahdanau Attention Mechanism**
adalah salah satu mekanisme perhatian (attention mechanism) yang digunakan dalam model sequence-to-sequence (seq2seq) untuk meningkatkan kinerja dalam tugas-tugas seperti penerjemahan bahasa, ringkasan teks, dan lain-lain. Mekanisme ini diperkenalkan oleh Dzmitry Bahdanau, Kyunghyun Cho, dan Yoshua Bengio pada tahun 2014.
- pertama buat scroing system =

Bhadanau = score -> normalize -> context

makanya return context

In [14]:
import tensorflow as tf

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__() # ini underscorenya 2x
        self.W1 = tf.keras.layers.Dense(units) # proyeksi state decoder (s(t-1)) ke dimensi attention
        self.W2 = tf.keras.layers.Dense(units) # proyeksi state encoder (h(t)) ke dimensi attention
        self.w_output = tf.keras.layers.Dense(1) # output

    # Rumus Pasti
    def call(self, prev_dec_state, enc_state): # ini rumus tanh(W1 * query + W2 * values) -> score -> softmax -> context vector
        scores = self.w_output(
            tf.nn.tanh(
                self.W1(
                    tf.expand_dims(prev_dec_state, -2) # (W1.s(t-1)) - disesuaikan ke self.W1
                ) +
                self.W2(
                    enc_state # (W2.h_1) - disesuaikan ke self.W2
                )
            )
        )
        
        # Normalize (indentasi disesuaikan)
        attention_weights = tf.nn.softmax(scores, 1) # softmax untuk normalisasi
        
        context_vector = attention_weights * enc_state # (16, 256)
        context_vector = tf.reduce_sum(context_vector, 1) # (256)

        return context_vector, attention_weights

# **Decoder**

In [19]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, sequence_length):
        super(Decoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)  # Kata indonesia embedding
        self.attention = BahdanauAttention(dec_units)  # Attention mechanism
        self.gru = tf.keras.layers.GRU(dec_units, return_sequences=True, return_state=True)  # GRU layer, RNN Decoder
        self.dense = tf.keras.layers.Dense(vocab_size)  # Output layer untuk prediksi kata berikutnya
        self.dec_units = dec_units
        self.sequence_length = sequence_length

    def call(self, x, shifted_target):
        outputs = []
        attention_weights = []
        shifted_target = self.embedding(shifted_target)  # simpan dalam batch (16, 256) . 256 = embedding dimensi
        hidden = tf.zeros((x.shape[0], self.dec_units))  # Inisialisasi hidden state GRU awal decoder = 0

        for t in range(0, self.sequence_length):  # Looping untuk setiap kata dalam sequence
            context_vector, attn_weights = self.attention(hidden, x)  # lihat lewat attention mechanism dulu baru tulis
            decoder_input = context_vector + shifted_target[:, t, :]  # supaya pindah yang baca
            output, hidden = self.gru(tf.expand_dims(decoder_input, 1), initial_state=hidden)  # GRU

            outputs.append(output[:, 0])  # simpan output
            attention_weights.append(attn_weights)  # simpan attention weights

        outputs = tf.convert_to_tensor(outputs)  # convert list ke tensor
        outputs = tf.transpose(outputs, perm=[1, 0, 2])  # transpose ke bentuk (batch_size, sequence_length, vocab_size)
        outputs = self.dense(outputs)  # prediksi kata berikutnya

        attention_weights = tf.convert_to_tensor(attention_weights)  # convert list ke tensor
        attention_weights = tf.transpose(attention_weights, perm=[1, 0, 2])  # transpose ke bentuk (batch_size, sequence_length, attention_weights)

        return outputs, attention_weights

# ini baru bikin kelasnya, kuis 2 beda lagi ini pokoknya ga masuk kuis.
# predict & repeat

In [30]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, sequence_length):
        super(Decoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.attention = BahdanauAttention(dec_units)
        self.gru = tf.keras.layers.GRU(dec_units, return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(vocab_size, activation="softmax")
        self.dec_units = dec_units
        self.sequence_length = sequence_length

    def call(self, x, shifted_target):
        outputs = []
        attention_weights = []
        shifted_target = self.embedding(shifted_target)
        
        # PENTING: Pakai tf.shape(x)[0] agar tidak TypeError/NoneType
        hidden = tf.zeros((tf.shape(x)[0], self.dec_units))

        for t in range(0, self.sequence_length):
            context_vector, attention_weight = self.attention(hidden, x)
            dec_inputs = context_vector + shifted_target[:, t]
            output, hidden = self.gru(tf.expand_dims(dec_inputs, 1), initial_state=hidden)

            outputs.append(output[:, 0])
            # PENTING: Ambil slice ini supaya dimensi pas saat di-transpose
            attention_weights.append(attention_weight[:, :, 0]) 

        outputs = tf.convert_to_tensor(outputs)
        outputs = tf.transpose(outputs, perm=[1, 0, 2])
        outputs = self.dense(outputs)

        attention_weights = tf.convert_to_tensor(attention_weights)
        attention_weights = tf.transpose(attention_weights, perm=[1, 0, 2])

        return outputs, attention_weights

In [ ]:
# # ini dibenerin dari gemini
# import tensorflow as tf

# class Decoder(tf.keras.layers.Layer): # Diubah ke Layer
    def __init__(self, vocab_size, embedding_dim, dec_units, sequence_length):
        super(Decoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)  # Kata indonesia embedding
        self.attention = BahdanauAttention(dec_units)  # Attention mechanism
        self.gru = tf.keras.layers.GRU(dec_units, return_sequences=True, return_state=True)  # GRU layer, RNN Decoder
        self.dense = tf.keras.layers.Dense(vocab_size)  # Output layer untuk prediksi kata berikutnya
        self.dec_units = dec_units
        self.sequence_length = sequence_length

    def call(self, x, shifted_target):
        outputs = []
        attention_weights = []
        shifted_target = self.embedding(shifted_target)  # (batch_size, sequence_length, embedding_dim)
        
        # PERBAIKAN 1: Gunakan tf.shape(x)[0] untuk dynamic batch size (None)
        batch_size = tf.shape(x)[0]
        hidden = tf.zeros((batch_size, self.dec_units))  # Inisialisasi hidden state GRU awal = 0

        for t in range(0, self.sequence_length):  # Looping untuk setiap kata dalam sequence
            context_vector, attn_weights = self.attention(hidden, x)  # lihat lewat attention mechanism
            decoder_input = context_vector + shifted_target[:, t, :]  # penjumlahan context & embedding kata ke-t
            output, hidden = self.gru(tf.expand_dims(decoder_input, 1), initial_state=hidden)  # GRU

            outputs.append(output[:, 0])  # simpan output
            attention_weights.append(attn_weights)  # simpan attention weights

        # PERBAIKAN 2: Gunakan tf.stack + tf.transpose untuk Keras Symbolic Tensor (Bukan tf.convert_to_tensor)
        outputs = tf.stack(outputs, axis=0)  # Shape: (sequence_length, batch_size, dec_units)
        outputs = tf.transpose(outputs, perm=[1, 0, 2])  # Shape: (batch_size, sequence_length, dec_units)
        outputs = self.dense(outputs)  # Prediksi kata berikutnya (batch_size, sequence_length, vocab_size)

        attention_weights = tf.stack(attention_weights, axis=0)  # Shape: (sequence_length, batch_size, enc_seq_len, 1)
        attention_weights = tf.transpose(attention_weights, perm=[1, 0, 2, 3])  # Shape: (batch_size, sequence_length, enc_seq_len, 1)

        return outputs, attention_weights

# **Making Model**

In [31]:
# Encoder
input_layer = tf.keras.layers.Input(shape=(ENG_SEQ_LENGTH,), name="input_1")
encoder = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_UNITS)
encoder_output = encoder(input_layer)

# Decoder
shifted_target = tf.keras.layers.Input(shape=(IND_SEQ_LENGTH,), name="input_2")
decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_UNITS, IND_SEQ_LENGTH)
decoder_output, attention_weights = decoder(encoder_output, shifted_target)

bahdanau = tf.keras.Model(inputs=[input_layer, shifted_target], outputs=decoder_output)
attention_model = tf.keras.Model(inputs=[input_layer, shifted_target], outputs=[decoder_output, attention_weights])

print(bahdanau.summary())

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_1             │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_8 (Encoder) │ (None, 16, 256)   │  2,317,312 │ input_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_2             │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_8 (Decoder) │ [(None, 16,       │  4,117,593 │ encoder_8[0][0],  │
│                     │ 7000), (None, 16, │            │ input_2[0][0]     │
│                     │ 16)]              │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,434,905 (24.55 MB)

 Trainable params: 6,434,905 (24.55 MB)

 Non-trainable params: 0 (0.00 B)

None


# **Training**

In [32]:
# Training
bahdanau.compile(optimizer="adam", 
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = bahdanau.fit(train_dataset, validation_data=val_dataset, epochs=20, callbacks=[early_stopping])

Epoch 1/20


c:\SEM 5\LAB AI Semester 5\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


KeyboardInterrupt: 

# **Result**

In [ ]:
result = bahdanau.evaluate(val_dataset)
print(f"Validation Loss: {result[0]}, Validation Accuracy: {result[1]}")